## Evaluations / Testing


### Setup

In [5]:
import os, json
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import torch
from torch.utils.data import DataLoader

from models import BaseClassifier
from train import test_model, build_class_to_top_class_mapping
from visualizations import LatentSpaceVisualizer, ConfusionMatrixVisualizer, ClassAccuracyAnalyzer, AttentionVisualizer, ActivationVisualizer
from dataset_utils import HATRDataset


BASE_PATH = "model_output/t-contr_ce_penalty/both/fold_0"
model_path = os.path.join(BASE_PATH, "best_model.pth")
latent_path = os.path.join(BASE_PATH, "latent_visualization")

with open(os.path.join(BASE_PATH, "history.json"), "r") as f:
    history = json.load(f) 
model_info = history.get('model_info', {})

model_name = model_info.get('hidden_size')
fold_id = model_info.get('fold_id')
hidden_size = model_info.get('hidden_size')
num_classes = model_info.get('num_classes')
emb_size_audio = model_info.get('emb_size_audio')
emb_size_text = model_info.get('emb_size_text')
dropout = model_info.get('dropout')
use_batch_norm = model_info.get('use_batch_norm')
mode = model_info.get('mode')
random_seed = model_info.get('random_seed')

class_dict_json = os.path.join('data', 'class_dict.json')
top_class_dict_json = os.path.join('data', 'top_class_dict.json')
with open(class_dict_json, 'r') as f:
    class_dict = json.load(f)
with open(top_class_dict_json, 'r') as f:
    top_class_dict = json.load(f)
class_to_top_class = build_class_to_top_class_mapping(class_dict, top_class_dict)

bsd10k_dataset = pd.read_csv(os.path.join('data', 'processed_dataset.csv'))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the model with the same architecture
model = BaseClassifier(hidden_size=hidden_size, num_classes=num_classes, emb_size_audio=emb_size_audio, emb_size_text=emb_size_text, 
                          dropout=dropout, use_batch_norm=use_batch_norm, mode=mode)

model.load_state_dict(torch.load(model_path))
model.eval()

BaseClassifier(
  (audio_emb_extractor): EmbeddingEncoder(
    (input_projection): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LeakyReLU(negative_slope=0.01)
      (2): Dropout(p=0.1, inplace=False)
    )
    (residual_blocks): ModuleList(
      (0-2): 3 x ResidualBlock(
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (linear2): Linear(in_features=1024, out_features=512, bias=True)
        (activation): LeakyReLU(negative_slope=0.01)
        (dropout): Dropout(p=0.1, inplace=False)
        (norm1): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (norm2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (output_projection): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): LeakyReLU(negative_slope=0.01)
      (2): Dropout(p=0.1, inplace=False)
      (3): Linear(in_features=256, out_fea

In [6]:
# Look history of model from JSON
print(history)

{'attention_audio': [0.501595, 0.497551, 0.499167, 0.50024, 0.497246, 0.49544, 0.497528, 0.499079, 0.498343, 0.497819, 0.495626, 0.494291, 0.50065, 0.499937, 0.502552, 0.50141, 0.502129, 0.501255, 0.499939, 0.502526, 0.507104, 0.50768, 0.505576, 0.505733, 0.507385, 0.508652, 0.507347, 0.509556, 0.506316, 0.506317, 0.507404, 0.506521, 0.508391, 0.50815, 0.507328, 0.509166, 0.510791, 0.508611, 0.507262, 0.507477, 0.507011, 0.506471], 'attention_text': [0.498405, 0.502449, 0.500833, 0.49976, 0.502754, 0.50456, 0.502472, 0.500921, 0.501657, 0.502181, 0.504374, 0.505709, 0.49935, 0.500063, 0.497448, 0.49859, 0.497871, 0.498745, 0.500061, 0.497474, 0.492896, 0.49232, 0.494424, 0.494267, 0.492615, 0.491348, 0.492653, 0.490444, 0.493684, 0.493683, 0.492596, 0.493479, 0.491609, 0.49185, 0.492672, 0.490834, 0.489209, 0.491389, 0.492738, 0.492523, 0.492989, 0.493529], 'train_cls_loss': [2.298621, 1.391428, 1.190903, 1.091573, 1.016069, 0.950443, 0.897299, 0.827517, 0.831934, 0.804289, 0.769542, 0

### Standalone testing

In [7]:
labels = bsd10k_dataset["class_idx"].tolist()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    if fold == fold_id:
        test_df = bsd10k_dataset.iloc[test_idx].reset_index()
        break  # stop once we reach the desired fold
    
test_dataset = HATRDataset(test_df, aug=False)
data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

accuracy, true_top_class_accuracy, acc_incl_second, top_class_acc_incl_second = test_model(
    model,
    model_path,
    data_loader,
    device,
    class_to_top_class,
    BASE_PATH,
    model_name,
    fold_id,
    class_dict=class_dict,
    top_class_dict=top_class_dict
)

print("Test complete!")

[128 | Fold 0] Accuracy: 79.61%
[128 | Fold 0] True Top Class Acc: 88.28%
[128 | Fold 0] Acc incl. 2nd: 91.42%
[128 | Fold 0] Top Class Acc incl. 2nd: 95.16%
Test complete!


In [8]:
labels = bsd10k_dataset["class_idx"].tolist()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1821)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    if fold == fold_id:
        test_df = bsd10k_dataset.iloc[test_idx].reset_index()
        break

test_dataset = HATRDataset(test_df, aug=False)
data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

# ---- Collect outputs ----
all_latents = []
all_logits = []
all_labels = []
all_sound_ids = []

with torch.no_grad():
    for batch in data_loader:
        labels_batch = batch['class_idx'].to(device)
        audio_emb = batch.get('audio_embedding', None)
        text_emb = batch.get('text_embedding', None)
        if audio_emb is not None: audio_emb = audio_emb.to(device)
        if text_emb is not None: text_emb = text_emb.to(device)

        z, logits, _ = model(audio_emb, text_emb)
        all_latents.append(z.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels_batch.cpu().numpy())
        all_sound_ids.extend(batch['sound_id'])

# ---- Flatten arrays ----
all_latents = np.concatenate(all_latents, axis=0)
all_logits = np.concatenate(all_logits, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# ---- Save latent vectors for PCA/t-SNE ----
np.save(os.path.join(BASE_PATH, f"latents.npy"), all_latents)


In [ ]:
# ---- Prepare test set from training ----
labels = bsd10k_dataset["class_idx"].tolist()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1821)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    if fold == fold_id:
        test_df = bsd10k_dataset.iloc[test_idx].reset_index(drop=True)
        break

test_dataset = HATRDataset(test_df, aug=False)
data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

# ---- Collect latents ----
all_latents, all_logits, all_labels, all_sound_ids = [], [], [], []
with torch.no_grad():
    for batch in data_loader:
        labels_batch = batch['class_idx'].to(device)
        audio_emb = batch.get('audio_embedding', None)
        text_emb = batch.get('text_embedding', None)
        if audio_emb is not None: audio_emb = audio_emb.to(device)
        if text_emb is not None: text_emb = text_emb.to(device)

        z, logits, _ = model(audio_emb, text_emb)

        all_latents.append(z.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels_batch.cpu().numpy())
        all_sound_ids.extend(batch['sound_id'])

all_latents = np.concatenate(all_latents, axis=0)
all_logits = np.concatenate(all_logits, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Save latents for visualization
latent_path = os.path.join(BASE_PATH, f"latents_fold{fold_id}.npy")
np.save(latent_path, all_latents)
np.save(os.path.join(BASE_PATH, f"labels_fold{fold_id}.npy"), all_labels)
np.save(os.path.join(BASE_PATH, f"sound_ids_fold{fold_id}.npy"), all_sound_ids)
np.save(os.path.join(BASE_PATH, f"logits_fold{fold_id}.npy"), all_logits)

print(f"Fold {fold_id} latents saved to {latent_path}")

Fold 0 latents saved to model_output/t-contr_ce_penalty/both/fold_0/latents_fold0.npy


In [ ]:
# ---- 3. Visualize ----
# TODO